# 03 Train NAND Best
Best-Path: Chars2Vec + SPECTER + InfoNCE, Architektur `818->1024->256`, Seeds stage-abhängig.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import sys

RUN_STAGE = "smoke"
RUN_ID = "replace_with_run_id_from_00"
DEVICE = "auto"
def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "configs").exists():
            return candidate.resolve()
    return start.resolve()

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.common.config import load_yaml
from src.common.io_schema import read_parquet
from src.approaches.nand.train import train_nand_across_seeds

model_cfg = load_yaml(PROJECT_ROOT / "configs/model/nand_best.yaml")
run_cfg = load_yaml(PROJECT_ROOT / f"configs/runs/{RUN_STAGE}.yaml")
training_cfg = model_cfg["training"]

subset_dir = PROJECT_ROOT / "data/subsets/cache" / RUN_ID
emb_dir = PROJECT_ROOT / "artifacts/embeddings" / RUN_ID
ckpt_dir = PROJECT_ROOT / "artifacts/checkpoints" / RUN_ID
metrics_dir = PROJECT_ROOT / "artifacts/metrics" / RUN_ID
ckpt_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

lspo_mentions = read_parquet(subset_dir / f"lspo_mentions_{RUN_STAGE}.parquet")
lspo_pairs = read_parquet(subset_dir / f"lspo_pairs_{RUN_STAGE}.parquet")
lspo_chars = np.load(emb_dir / f"lspo_chars2vec_{RUN_STAGE}.npy")
lspo_text = np.load(emb_dir / f"lspo_specter_{RUN_STAGE}.npy")

print("Model:", model_cfg.get("name"))
print("Loss:", training_cfg.get("loss"))
print("Architecture:", f"{training_cfg['input_dim']} -> {training_cfg['hidden_dim']} -> {training_cfg['output_dim']}")


In [ ]:
# Stage-seed policy: smoke/mini 1 seed, mid/full 5 seeds
default_seeds = training_cfg.get("seeds", [1,2,3,4,5])
if RUN_STAGE in {"smoke", "mini"}:
    seeds = [default_seeds[0]]
else:
    seeds = default_seeds

manifest_path = metrics_dir / "03_train_manifest.json"
manifest = train_nand_across_seeds(
    mentions=lspo_mentions,
    pairs=lspo_pairs,
    chars2vec=lspo_chars,
    text_emb=lspo_text,
    model_config=training_cfg,
    seeds=seeds,
    run_id=RUN_ID,
    output_dir=ckpt_dir,
    metrics_output=manifest_path,
    device=DEVICE,
)

manifest


In [ ]:
# Train/Val/Test summary
rows = []
for r in manifest["runs"]:
    rows.append({
        "seed": r["seed"],
        "checkpoint": r["checkpoint"],
        "threshold": r["threshold"],
        "val_f1": r["val_stats"].get("f1"),
        "val_precision": r["val_stats"].get("precision"),
        "val_recall": r["val_stats"].get("recall"),
        "test_f1": r["test_metrics"].get("f1"),
        "test_precision": r["test_metrics"].get("precision"),
        "test_recall": r["test_metrics"].get("recall"),
    })

pd.DataFrame(rows).sort_values("val_f1", ascending=False)
